In [45]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy import units as u
from astropy import constants as const

In [14]:
df = pd.read_csv("solar_system.csv")

In [15]:
df = df.set_index("Attribute").T
df.index.name = "Planet"
df.reset_index(inplace=True)
df.columns.name = None

In [16]:
df.shape

(10, 21)

The transposed dataframe consists of 10 rows and 21 columns. The original dataframe thus consists of 21 rows and 10 columns.

The first line of code sets 'Attribute' as an index as transposes the dataframe.
The second line of code names the new index 'Planet'.
The third line of code converts the 'Planet' index back into a column.
The last line of code removes the 'Attribute' name from columns.
The new shape isn't 11, 21 because we remove a row.

In [17]:
print(df.columns)

Index(['Planet', 'Mass (10^24kg)', 'Diameter (km)', 'Density (kg/m^3)',
       'Gravity (m/s^2)', 'Escape Velocity (km/s)', 'Rotation Period (hours)',
       'Length of Day (hours)', 'Distance from Sun (10^6 km)',
       'Perihelion (10^6 km)', 'Aphelion (10^6 km)', 'Orbital Period (days)',
       'Orbital Velocity (km/s)', 'Orbital Inclination (deg)',
       'Orbital Eccentricity', 'Obliquity to Orbit (deg)',
       'Mean Temperature (C)', 'Surface Pressure (bars)', 'Number of Moons',
       'Ring System?', 'Global Magnetic Field?'],
      dtype='object')


16 columns have units, while 5 do not.

In [9]:
print(df)

    Planet Mass (10^24kg) Diameter (km) Density (kg/m^3) Gravity (m/s^2)  \
0  Mercury          0.330          4879             5429             3.7   
1    Venus           4.87         12104             5243             8.9   
2    Earth           5.97         12756             5514             9.8   
3     Moon          0.073          3475             3340             1.6   
4     Mars          0.642          6792             3934             3.7   
5  Jupiter           1898        142984             1326            23.1   
6   Saturn            568        120536              687             9.0   
7   Uranus           86.8         51118             1270             8.7   
8  Neptune            102         49528             1638            11.0   
9    Pluto         0.0130          2376             1850             0.7   

  Escape Velocity (km/s) Rotation Period (hours) Length of Day (hours)  \
0                    4.3                  1407.6                4222.6   
1              

In [20]:
df.dtypes

Planet                         object
Mass (10^24kg)                 object
Diameter (km)                  object
Density (kg/m^3)               object
Gravity (m/s^2)                object
Escape Velocity (km/s)         object
Rotation Period (hours)        object
Length of Day (hours)          object
Distance from Sun (10^6 km)    object
Perihelion (10^6 km)           object
Aphelion (10^6 km)             object
Orbital Period (days)          object
Orbital Velocity (km/s)        object
Orbital Inclination (deg)      object
Orbital Eccentricity           object
Obliquity to Orbit (deg)       object
Mean Temperature (C)           object
Surface Pressure (bars)        object
Number of Moons                object
Ring System?                   object
Global Magnetic Field?         object
dtype: object

In [21]:
for col in df:
    print(col, df[col].apply(type).value_counts())

Planet Planet
<class 'str'>    10
Name: count, dtype: int64
Mass (10^24kg) Mass (10^24kg)
<class 'str'>    10
Name: count, dtype: int64
Diameter (km) Diameter (km)
<class 'str'>    10
Name: count, dtype: int64
Density (kg/m^3) Density (kg/m^3)
<class 'str'>    10
Name: count, dtype: int64
Gravity (m/s^2) Gravity (m/s^2)
<class 'str'>    10
Name: count, dtype: int64
Escape Velocity (km/s) Escape Velocity (km/s)
<class 'str'>    10
Name: count, dtype: int64
Rotation Period (hours) Rotation Period (hours)
<class 'str'>    10
Name: count, dtype: int64
Length of Day (hours) Length of Day (hours)
<class 'str'>    10
Name: count, dtype: int64
Distance from Sun (10^6 km) Distance from Sun (10^6 km)
<class 'str'>    10
Name: count, dtype: int64
Perihelion (10^6 km) Perihelion (10^6 km)
<class 'str'>    10
Name: count, dtype: int64
Aphelion (10^6 km) Aphelion (10^6 km)
<class 'str'>    10
Name: count, dtype: int64
Orbital Period (days) Orbital Period (days)
<class 'str'>    10
Name: count, dtype

a. The actual values are strings. \
b. Storing numbers as strings can be problematic for many reasons, but one of the most significant issues is that strings are sorted alphabetically. For instance, 10 (ten) would be ordered before 2 (two).

In [31]:
for col in df:
    if col not in ['Planet', 'Ring System?', 'Global Magnetic Field?']:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [32]:
df.dtypes

Planet                          object
Mass (10^24kg)                 float64
Diameter (km)                    int64
Density (kg/m^3)                 int64
Gravity (m/s^2)                float64
Escape Velocity (km/s)         float64
Rotation Period (hours)        float64
Length of Day (hours)          float64
Distance from Sun (10^6 km)    float64
Perihelion (10^6 km)           float64
Aphelion (10^6 km)             float64
Orbital Period (days)          float64
Orbital Velocity (km/s)        float64
Orbital Inclination (deg)      float64
Orbital Eccentricity           float64
Obliquity to Orbit (deg)       float64
Mean Temperature (C)             int64
Surface Pressure (bars)        float64
Number of Moons                  int64
Ring System?                    object
Global Magnetic Field?          object
dtype: object

a. The three data types are objects, floats, and integers. \
b. A float is any numerical value containing a decimal point. An integer is a numerical value that does not contain a decimal point. The last data type is a string, which can contain words or numbers. \
c. The three columns that are still classified as 'object' are as such because their entries do not contain numerical values, but rather words.

In [33]:
def attach_units(col, unit, new_col=None):
    """
    Applies an Astropy unit to a DataFrame column and optionally renames it.
    
    Parameters:
        col     : str   - existing column name in df
        unit    : astropy.units - the unit to apply (e.g. 1e24 * u.kg)
        new_col : str   - new column name (optional, skips rename if None)
    """
    # Apply the unit to each value in the column
    df[col] = [val * unit for val in df[col]]
    
    # Rename the column if a new name is provided
    if new_col:
        df.rename(columns={col: new_col}, inplace=True)    

In [35]:
attach_units("Diameter (km)",  u.km, new_col="Diameter (km)")

In [36]:
df['Diameter (km)']

0      4879.0 km2
1     12104.0 km2
2     12756.0 km2
3      3475.0 km2
4      6792.0 km2
5    142984.0 km2
6    120536.0 km2
7     51118.0 km2
8     49528.0 km2
9      2376.0 km2
Name: Diameter (km), dtype: object

In [40]:
df['Aphelion (10^6 km)']

0      69.800
1     108.900
2     152.100
3       0.406
4     249.300
5     816.400
6    1506.500
7    3001.400
8    4558.900
9    7375.900
Name: Aphelion (10^6 km), dtype: float64

In [43]:
semi_major = (df['Perihelion (10^6 km)'] + df['Aphelion (10^6 km)']) / 2

aphelion_idx = df.columns.get_loc('Aphelion (10^6 km)')
df.insert(aphelion_idx + 1, "Semi-Major Axis (10^6 km)", semi_major)

print(df.columns.tolist())

print(df['Semi-Major Axis (10^6 km)'])

['Planet', 'Mass (10^24kg)', 'Diameter (km)', 'Density (kg/m^3)', 'Gravity (m/s^2)', 'Escape Velocity (km/s)', 'Rotation Period (hours)', 'Length of Day (hours)', 'Distance from Sun (10^6 km)', 'Perihelion (10^6 km)', 'Aphelion (10^6 km)', 'Semi-Major Axis (10^6 km)', 'Orbital Period (days)', 'Orbital Velocity (km/s)', 'Orbital Inclination (deg)', 'Orbital Eccentricity', 'Obliquity to Orbit (deg)', 'Mean Temperature (C)', 'Surface Pressure (bars)', 'Number of Moons', 'Ring System?', 'Global Magnetic Field?']
0      57.9000
1     108.2000
2     149.6000
3       0.3845
4     228.0000
5     778.5000
6    1432.0500
7    2867.0500
8    4515.0000
9    5906.3500
Name: Semi-Major Axis (10^6 km), dtype: float64


In [47]:
# Convert Orbital Period to Years
df["Orbital Period (days)"] = df["Orbital Period (days)"].apply(lambda val: (val * u.day).to(u.yr))
    # lambda val: val.to(u.yr) converts every cell in the column to the value of interest

df.rename(columns={"Orbital Period (days)": "Orbital Period (yr)"}, inplace=True)

In [48]:
print(df.columns.tolist())


['Planet', 'Mass (10^24kg)', 'Diameter (km)', 'Density (kg/m^3)', 'Gravity (m/s^2)', 'Escape Velocity (km/s)', 'Rotation Period (hours)', 'Length of Day (hours)', 'Distance from Sun (10^6 km)', 'Perihelion (10^6 km)', 'Aphelion (10^6 km)', 'Semi-Major Axis (10^6 km)', 'Orbital Period (yr)', 'Orbital Velocity (km/s)', 'Orbital Inclination (deg)', 'Orbital Eccentricity', 'Obliquity to Orbit (deg)', 'Mean Temperature (C)', 'Surface Pressure (bars)', 'Number of Moons', 'Ring System?', 'Global Magnetic Field?']


In [49]:
print(df["Orbital Period (yr)"])

0    0.24093086926762491 yr
1     0.6151950718685831 yr
2     0.9998631074606433 yr
3    0.07474332648870637 yr
4     1.8809034907597535 yr
5      11.85763175906913 yr
6     29.423682409308693 yr
7      83.74811772758385 yr
8     163.72347707049965 yr
9     247.93976728268308 yr
Name: Orbital Period (yr), dtype: object


In [52]:
planet = "Earth"
period = df.loc[df["Planet"] == planet, "Orbital Period (yr)"].values[0]
print(f"{planet}'s orbital period: {period:.4f}")

Earth's orbital period: 0.9999 yr


An astronomical unit (abbreviated AU) is the distance from the Sun to Earth. It is approximately 1.5 * 10^11 m.

In [53]:
print(const.au)

  Name   = Astronomical Unit
  Value  = 149597870700.0
  Uncertainty  = 0.0
  Unit  = m
  Reference = IAU 2012 Resolution B2


In [55]:
au = const.au
au.to(u.km)

<Quantity 1.49597871e+08 km>

In [64]:
for col in distance_cols:
    val = df[col].dropna().values[0]
    print(col, "-->", type(val), val)

Diameter (km) --> <class 'astropy.units.quantity.Quantity'> 4879.0 km2
Distance from Sun (10^6 km) --> <class 'numpy.float64'> 57.9
Perihelion (10^6 km) --> <class 'numpy.float64'> 46.0
Aphelion (10^6 km) --> <class 'numpy.float64'> 69.8
Semi-Major Axis (10^6 km) --> <class 'numpy.float64'> 57.9


In [65]:
for old_col, new_col in distance_cols.items():
    def convert(val):
        if not pd.notnull(val): 
            return val
        # Strip existing astropy units if already attached
        if hasattr(val, 'unit'):
            val = val.value  # get raw float
        if "10^6" in old_col:
            return (val * 1e6 * u.km).to(u.AU)
        else:
            return (val * u.km).to(u.AU)
    
    df[new_col] = df[old_col].apply(convert)
    df.drop(columns=[old_col], inplace=True)

print(df.columns.tolist())

['Planet', 'Mass (10^24kg)', 'Density (kg/m^3)', 'Gravity (m/s^2)', 'Escape Velocity (km/s)', 'Rotation Period (hours)', 'Length of Day (hours)', 'Orbital Period (yr)', 'Orbital Velocity (km/s)', 'Orbital Inclination (deg)', 'Orbital Eccentricity', 'Obliquity to Orbit (deg)', 'Mean Temperature (C)', 'Surface Pressure (bars)', 'Number of Moons', 'Ring System?', 'Global Magnetic Field?', 'Diameter (AU)', 'Distance from Sun (AU)', 'Perihelion (AU)', 'Aphelion (AU)', 'Semi-Major Axis (AU)']


In [67]:
planet = "Earth"
earth = df[df["Planet"] == planet]

for col in ["Diameter (AU)", "Distance from Sun (AU)", "Perihelion (AU)", "Aphelion (AU)", "Semi-Major Axis (AU)"]:
    val = earth[col].values[0]
    print(f"{col}: {val:.6f}")

Diameter (AU): 0.000085 AU
Distance from Sun (AU): 1.000014 AU
Perihelion (AU): 0.983303 AU
Aphelion (AU): 1.016726 AU
Semi-Major Axis (AU): 1.000014 AU


In [68]:
df.to_csv('units.csv', index = False)